# 01 — Scene detection bằng TransNetV2
#
Notebook xử lý **tất cả video trong raw folder** và chỉ tạo dữ liệu scene.
#
**Input bắt buộc:** folder video raw.
**Input tùy chọn:** `video_manifest.jsonl` từ notebook 00.
#
**Output:**
#
- `manifests/scene_manifest.jsonl`
- `videos/<video_id>/scene_manifest.jsonl`
- stage signature và `_SUCCESS.json`
- `stage_timings.jsonl` — thời gian chạy của **từng tác vụ** (từng video + từng bước pipeline)
- `01_scene_detection_output.zip`
#
Notebook không tự fallback sang fixed-window. Nếu muốn fallback, phải bật rõ cấu hình.
#
## Cách notebook này được tổ chức
#
Pipeline được chia thành các **tác vụ (task) độc lập, có tên rõ ràng**:
#
1. `setup_output`         — tạo thư mục output
2. `discover_videos`      — tìm / đọc danh sách video cần xử lý
3. `resolve_weights`      — tìm file trọng số TransNetV2
4. `load_model`           — load model lên GPU/CPU
5. `video:<video_id>`     — một tác vụ riêng cho **mỗi video** (lặp lại N lần)
6. `write_manifests`      — ghi manifest tổng hợp + model_info
7. `package_zip`          — đóng gói toàn bộ output
8. `validate_output`      — kiểm tra tính hợp lệ của kết quả
#
Mỗi tác vụ đều được đo thời gian, log rõ lúc bắt đầu/kết thúc, và ghi vào
`progress.json` để theo dõi trạng thái real-time. Cuối notebook có bảng tổng kết
thời gian chạy của toàn bộ tác vụ, giúp đánh giá tác vụ nào tốn thời gian nhất.

In [1]:
%pip install -q --no-cache-dir "transnetv2-pytorch==1.0.5" "opencv-python-headless>=4.9,<5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.7/32.7 MB 105.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## 1. Cấu hình

In [2]:
from pathlib import Path
import os

RAW_VIDEO_ROOT = Path(os.environ.get(
    "AIC_RAW_VIDEO_ROOT",
    "/kaggle/input/datasets/trongnhantran25/aic-nam-thang-ay/Videos_L21_a/video"
))
OUTPUT_ROOT = Path(os.environ.get(
    "AIC_STAGE01_OUTPUT",
    "/kaggle/working/aic_stage_01_scenes"
))

VIDEO_MANIFEST_PATH = Path(os.environ.get(
    "AIC_VIDEO_MANIFEST", ""
)) if os.environ.get("AIC_VIDEO_MANIFEST", "") else None

TRANSNET_WEIGHTS_PATH = Path(os.environ.get(
    "AIC_TRANSNET_WEIGHTS", ""
)) if os.environ.get("AIC_TRANSNET_WEIGHTS", "") else None

THRESHOLD = 0.50
MAX_VIDEOS = 0                 # 0 = tất cả
MAX_SCENES_PER_VIDEO = 0
ALLOW_NETWORK_DOWNLOAD = False
ALLOW_FIXED_WINDOW_FALLBACK = False
FIXED_WINDOW_SEC = 8.0
PACK_VERSION = "1.1.0"

print("RAW_VIDEO_ROOT =", RAW_VIDEO_ROOT)
print("OUTPUT_ROOT    =", OUTPUT_ROOT)

RAW_VIDEO_ROOT = /kaggle/input/datasets/trongnhantran25/aic-nam-thang-ay/Videos_L21_a/video
OUTPUT_ROOT    = /kaggle/working/aic_stage_01_scenes


## 2. Theo dõi tiến độ & đo thời gian từng tác vụ
#
- `TaskTracker.stage(name)` — context manager: log bắt đầu / kết thúc / lỗi, đo thời
  gian, ghi lại vào danh sách `records` để tổng kết cuối notebook.
- `progress.json` — trạng thái hiện tại (tác vụ nào đang chạy, video thứ mấy, ETA...).
- `progress.log` — log dạng text, có heartbeat khi một tác vụ chạy lâu (ví dụ TransNetV2
  đang xử lý một video dài) để biết notebook không bị treo.

In [3]:
SHOW_TQDM = True
HEARTBEAT_INTERVAL_SEC = 20
WRITE_PROGRESS_JSON = True
PROGRESS_JSON_PATH = OUTPUT_ROOT / "progress.json"
PROGRESS_LOG_PATH = OUTPUT_ROOT / "progress.log"
STAGE_TIMINGS_PATH = OUTPUT_ROOT / "stage_timings.jsonl"

print("SHOW_TQDM              =", SHOW_TQDM)
print("HEARTBEAT_INTERVAL_SEC =", HEARTBEAT_INTERVAL_SEC)
print("PROGRESS_JSON_PATH     =", PROGRESS_JSON_PATH)

SHOW_TQDM              = True
HEARTBEAT_INTERVAL_SEC = 20
PROGRESS_JSON_PATH     = /kaggle/working/aic_stage_01_scenes/progress.json


In [4]:
import hashlib
import inspect
import json
import os
import threading
import time
import urllib.request
import zipfile
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from typing import Any, Optional

import cv2
import torch
from tqdm.auto import tqdm
from transnetv2_pytorch import TransNetV2

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v"}


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def append_progress_log(message: str) -> None:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    line = f"[{utc_now()}] {message}"
    print(line, flush=True)
    with PROGRESS_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def atomic_write_bytes(path: Path, data: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    with temp.open("wb") as handle:
        handle.write(data)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temp, path)


def atomic_write_json(path: Path, payload: Any) -> None:
    atomic_write_bytes(path, (json.dumps(payload, ensure_ascii=False, indent=2) + "\n").encode("utf-8"))


def atomic_write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    content = "".join(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n" for row in rows)
    atomic_write_bytes(path, content.encode("utf-8"))


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def write_progress_state(payload: dict[str, Any]) -> None:
    if not WRITE_PROGRESS_JSON:
        return
    state = {"stage": "01_scene_detection", "updated_at_utc": utc_now(), **payload}
    atomic_write_json(PROGRESS_JSON_PATH, state)


# ---------------------------------------------------------------------------
# TaskTracker: mỗi tác vụ (task) là một "stage" có tên, có start/end/duration
# ---------------------------------------------------------------------------
@dataclass
class StageRecord:
    name: str
    start_ts: float
    end_ts: Optional[float] = None
    status: str = "running"
    detail: str = ""

    @property
    def duration_sec(self) -> float:
        end = self.end_ts if self.end_ts is not None else time.perf_counter()
        return end - self.start_ts


class TaskTracker:
    """Theo dõi tiến độ & thời gian chạy của từng tác vụ trong pipeline."""

    def __init__(self) -> None:
        self.records: list[StageRecord] = []
        self._pipeline_started = time.perf_counter()

    @contextmanager
    def stage(self, name: str, detail: str = ""):
        record = StageRecord(name=name, start_ts=time.perf_counter(), detail=detail)
        self.records.append(record)
        append_progress_log(f"▶ BẮT ĐẦU  [{name}]" + (f" — {detail}" if detail else ""))
        write_progress_state({"status": "running", "current_task": name,
                               "elapsed_sec": round(time.perf_counter() - self._pipeline_started, 3)})
        try:
            yield record
            record.status = "success"
        except Exception as exc:
            record.status = "failed"
            record.end_ts = time.perf_counter()
            append_progress_log(f"✖ LỖI      [{name}] sau {record.duration_sec:.1f}s | {exc!r}")
            append_jsonl(STAGE_TIMINGS_PATH, self._record_to_row(record))
            write_progress_state({"status": "failed", "current_task": name, "error": repr(exc)})
            raise
        else:
            record.end_ts = time.perf_counter()
            suffix = f" — {record.detail}" if record.detail else ""
            append_progress_log(f"✔ HOÀN TẤT [{name}] sau {record.duration_sec:.1f}s{suffix}")
            append_jsonl(STAGE_TIMINGS_PATH, self._record_to_row(record))

    @staticmethod
    def _record_to_row(record: StageRecord) -> dict[str, Any]:
        return {
            "task": record.name,
            "status": record.status,
            "duration_sec": round(record.duration_sec, 3),
            "detail": record.detail,
            "finished_at_utc": utc_now(),
        }

    def elapsed_total_sec(self) -> float:
        return time.perf_counter() - self._pipeline_started

    def summary_rows(self) -> list[dict[str, Any]]:
        return [self._record_to_row(r) for r in self.records]


@contextmanager
def heartbeat(label: str, interval_sec: int = 20):
    """In log định kỳ trong khi một tác vụ dài đang chạy, để biết notebook chưa bị treo."""
    stop_event = threading.Event()
    start_time = time.perf_counter()

    def worker() -> None:
        while not stop_event.wait(interval_sec):
            elapsed = time.perf_counter() - start_time
            append_progress_log(f"   …vẫn đang chạy: {label} | elapsed={elapsed/60:.1f} phút")

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    try:
        yield
    finally:
        stop_event.set()
        thread.join(timeout=1)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def create_zip(folder: Path, zip_path: Path) -> Path:
    temp = zip_path.with_name(zip_path.name + ".tmp")
    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(folder))
    os.replace(temp, zip_path)
    return zip_path


def discover_videos(root: Path) -> list[Path]:
    videos = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)
    if not videos:
        raise FileNotFoundError(f"No raw videos under {root}")
    return videos[:MAX_VIDEOS] if MAX_VIDEOS > 0 else videos


def probe_minimal(path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    try:
        fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    finally:
        capture.release()
    return {
        "video_id": path.stem,
        "source_path": str(path),
        "fps": fps,
        "frame_count": frame_count,
        "checksum_sha256": sha256_file(path),
    }


def load_videos() -> list[dict[str, Any]]:
    if VIDEO_MANIFEST_PATH and VIDEO_MANIFEST_PATH.exists():
        rows = read_jsonl(VIDEO_MANIFEST_PATH)
        selected = rows[:MAX_VIDEOS] if MAX_VIDEOS > 0 else rows
        for row in selected:
            if not Path(row["source_path"]).exists():
                candidate = RAW_VIDEO_ROOT / row.get("source_relative_path", "")
                if candidate.exists():
                    row["source_path"] = str(candidate)
        return selected
    return [probe_minimal(path) for path in discover_videos(RAW_VIDEO_ROOT)]


def signature(video: dict[str, Any]) -> str:
    payload = {
        "stage": "scene_detection",
        "pack_version": PACK_VERSION,
        "video_id": video["video_id"],
        "video_checksum": video["checksum_sha256"],
        "threshold": THRESHOLD,
        "max_scenes": MAX_SCENES_PER_VIDEO,
        "fallback": ALLOW_FIXED_WINDOW_FALLBACK,
        "fixed_window_sec": FIXED_WINDOW_SEC,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()

## 3. Resolve trọng số & chuẩn hóa scene liên tục

In [5]:
import transnetv2_pytorch


def resolve_weights() -> Path:
    candidates: list[Path] = []
    if TRANSNET_WEIGHTS_PATH:
        candidates.append(TRANSNET_WEIGHTS_PATH)
    candidates.extend(Path("/kaggle/input").rglob("transnetv2-pytorch-weights.pth"))
    package_root = Path(inspect.getfile(transnetv2_pytorch)).parent
    candidates.extend(package_root.rglob("transnetv2-pytorch-weights.pth"))
    candidates.extend(package_root.rglob("*.pth"))

    for candidate in candidates:
        if candidate.is_file() and candidate.stat().st_size > 1_000_000:
            return candidate

    if ALLOW_NETWORK_DOWNLOAD:
        destination = Path("/kaggle/working/transnetv2-pytorch-weights.pth")
        urllib.request.urlretrieve(
            "https://huggingface.co/Sn4kehead/TransNetV2/resolve/main/transnetv2-pytorch-weights.pth",
            destination,
        )
        return destination

    raise FileNotFoundError(
        "TransNetV2 weights not found. Add the verified .pth file as a Kaggle "
        "Dataset or set ALLOW_NETWORK_DOWNLOAD=True explicitly."
    )


def parse_raw_scene(item: Any) -> tuple[int, int, float]:
    if isinstance(item, dict):
        start = int(item.get("start_frame", item.get("start_frame_index", 0)))
        end = int(item.get("end_frame", item.get("end_frame_index", start)))
        confidence = float(item.get("confidence", item.get("score", 0.0)) or 0.0)
        return start, end, confidence
    values = list(item)
    if len(values) < 2:
        raise ValueError(f"Unexpected scene output: {item!r}")
    return int(values[0]), int(values[1]), float(values[2]) if len(values) > 2 else 0.0


def normalize_contiguous(raw: Any, frame_count: int) -> list[tuple[int, int, float, bool]]:
    if frame_count <= 0:
        return []
    parsed = []
    for item in raw:
        start, end, confidence = parse_raw_scene(item)
        start = max(0, min(start, frame_count - 1))
        end = max(start, min(end, frame_count - 1))
        parsed.append((start, end, confidence))
    parsed.sort(key=lambda row: (row[0], row[1]))

    normalized: list[tuple[int, int, float, bool]] = []
    cursor = 0
    for start, end, confidence in parsed:
        if end < cursor:
            continue
        if start > cursor:
            normalized.append((cursor, start - 1, 0.0, True))
        start = max(start, cursor)
        normalized.append((start, end, confidence, start != parsed[0][0] and start == cursor))
        cursor = end + 1
        if cursor >= frame_count:
            break
    if cursor < frame_count:
        normalized.append((cursor, frame_count - 1, 0.0, True))
    if not normalized:
        normalized = [(0, frame_count - 1, 0.0, True)]

    cleaned = []
    cursor = 0
    for start, end, confidence, repaired in normalized:
        start = cursor
        end = max(start, min(end, frame_count - 1))
        cleaned.append((start, end, confidence, repaired))
        cursor = end + 1
        if cursor >= frame_count:
            break
    if cleaned[-1][1] < frame_count - 1:
        cleaned.append((cleaned[-1][1] + 1, frame_count - 1, 0.0, True))
    return cleaned


def fixed_windows(frame_count: int, fps: float) -> list[tuple[int, int, float, bool]]:
    size = max(1, int(round(FIXED_WINDOW_SEC * fps)))
    rows = []
    start = 0
    while start < frame_count:
        end = min(frame_count - 1, start + size - 1)
        rows.append((start, end, 0.0, True))
        start = end + 1
    return rows

## 4. Pipeline chính
#
Mỗi bước dưới đây là **một tác vụ riêng biệt**, chạy dưới `tracker.stage(...)`.
Video cũng được xử lý như từng tác vụ con (`video:<id>`), để có thể theo dõi
và so sánh thời gian xử lý của từng video riêng lẻ.

In [6]:
tracker = TaskTracker()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if STAGE_TIMINGS_PATH.exists():
    STAGE_TIMINGS_PATH.unlink()

# ---- Task 1: setup output ----
with tracker.stage("setup_output"):
    (OUTPUT_ROOT / "manifests").mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / "videos").mkdir(parents=True, exist_ok=True)

[2026-07-31T20:51:45.030682+00:00] ▶ BẮT ĐẦU  [setup_output]
[2026-07-31T20:51:45.038109+00:00] ✔ HOÀN TẤT [setup_output] sau 0.0s


In [7]:
# ---- Task 2: discover / load danh sách video ----
with tracker.stage("discover_videos") as rec:
    videos = load_videos()
    rec.detail = f"{len(videos)} video"

[2026-07-31T20:51:45.052835+00:00] ▶ BẮT ĐẦU  [discover_videos]
[2026-07-31T20:52:08.276277+00:00] ✔ HOÀN TẤT [discover_videos] sau 23.2s — 29 video


In [8]:
# ---- Task 3: resolve trọng số TransNetV2 ----
with tracker.stage("resolve_weights") as rec:
    weights = resolve_weights()
    weights_checksum = sha256_file(weights)
    rec.detail = str(weights)

[2026-07-31T20:52:08.293288+00:00] ▶ BẮT ĐẦU  [resolve_weights]
[2026-07-31T21:00:50.137036+00:00] ✔ HOÀN TẤT [resolve_weights] sau 521.8s — /usr/local/lib/python3.12/dist-packages/transnetv2_pytorch/transnetv2-pytorch-weights.pth


In [9]:
# ---- Task 4: load model lên device ----
device = "cuda:0" if torch.cuda.is_available() else "cpu"
with tracker.stage("load_model", detail=f"device={device}"):
    model = TransNetV2(device=device)
    try:
        state = torch.load(str(weights), map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(str(weights), map_location=device)
    model.load_state_dict(state)
    model.eval()

[2026-07-31T21:00:50.427740+00:00] ▶ BẮT ĐẦU  [load_model] — device=cuda:0
[2026-07-31T21:00:55.640002+00:00] ✔ HOÀN TẤT [load_model] sau 5.2s — device=cuda:0


### 4.1 Xử lý từng video
#
Mỗi video là một tác vụ (`video:<video_id>`) với thanh tiến độ tổng
(số video đã xong / tổng số), ETA ước tính dựa trên thời gian trung bình
của các video đã xử lý, và heartbeat khi TransNetV2 đang chạy lâu.

In [10]:
all_rows: list[dict[str, Any]] = []
stage_rows: list[dict[str, Any]] = []

video_bar = tqdm(total=len(videos), desc="Scene detection", unit="video",
                  dynamic_ncols=True, disable=not SHOW_TQDM)

completed_durations: list[float] = []

try:
    for video_number, video in enumerate(videos, start=1):
        video_id = str(video["video_id"])
        video_path = Path(video["source_path"])
        video_output = OUTPUT_ROOT / "videos" / video_id
        video_output.mkdir(parents=True, exist_ok=True)
        stage_sig = signature(video)

        avg_runtime = sum(completed_durations) / len(completed_durations) if completed_durations else 0.0
        eta_sec = avg_runtime * (len(videos) - (video_number - 1))
        video_bar.set_postfix_str(f"{video_id} | scenes={len(all_rows)} | ETA={eta_sec/60:.1f}m")

        write_progress_state({
            "status": "running",
            "current_task": f"video:{video_id}",
            "video_total": len(videos),
            "video_completed": video_number - 1,
            "current_video_number": video_number,
            "current_video_id": video_id,
            "scene_count_total": len(all_rows),
            "elapsed_sec": round(tracker.elapsed_total_sec(), 3),
            "eta_sec": round(eta_sec, 3),
        })

        error = ""
        with tracker.stage(f"video:{video_id}",
                            detail=f"{video_number}/{len(videos)} frames={video.get('frame_count', 0)}") as rec:
            try:
                with heartbeat(f"TransNetV2 đang xử lý {video_id}", HEARTBEAT_INTERVAL_SEC):
                    with torch.no_grad():
                        raw = model.detect_scenes(str(video_path), threshold=THRESHOLD)
                scenes = normalize_contiguous(raw, int(video["frame_count"]))
                backend = "transnetv2_pytorch"
            except Exception as exc:
                if not ALLOW_FIXED_WINDOW_FALLBACK:
                    raise
                append_progress_log(f"[WARN] {video_id}: TransNetV2 lỗi, dùng fixed-window fallback")
                scenes = fixed_windows(int(video["frame_count"]), float(video["fps"]))
                backend = "fixed_window_explicit_fallback"
                error = repr(exc)

            if MAX_SCENES_PER_VIDEO > 0:
                scenes = scenes[:MAX_SCENES_PER_VIDEO]

            fps = float(video["fps"])
            rows = []
            for scene_index, (start, end, confidence, repaired) in enumerate(scenes):
                rows.append({
                    "video_id": video_id,
                    "scene_id": f"{video_id}_S{scene_index:05d}",
                    "scene_index": scene_index,
                    "start_frame": int(start),
                    "end_frame": int(end),
                    "start_sec": float(start / fps) if fps > 0 else 0.0,
                    "end_sec": float(end / fps) if fps > 0 else 0.0,
                    "duration_sec": float((end - start + 1) / fps) if fps > 0 else 0.0,
                    "detector": backend,
                    "detector_threshold": float(THRESHOLD),
                    "confidence": float(confidence),
                    "boundary_repaired": bool(repaired),
                    "stage_signature": stage_sig,
                })

            for previous, current in zip(rows, rows[1:]):
                assert previous["end_frame"] + 1 == current["start_frame"]
            if rows and MAX_SCENES_PER_VIDEO == 0:
                assert rows[0]["start_frame"] == 0
                assert rows[-1]["end_frame"] == int(video["frame_count"]) - 1

            atomic_write_jsonl(video_output / "scene_manifest.jsonl", rows)
            rec.detail = f"scenes={len(rows)} backend={backend}"

        video_runtime = tracker.records[-1].duration_sec
        completed_durations.append(video_runtime)

        atomic_write_json(video_output / "_SUCCESS.json", {
            "status": "success",
            "video_id": video_id,
            "scene_count": len(rows),
            "stage_signature": stage_sig,
            "backend": backend,
            "runtime_sec": round(video_runtime, 3),
            "error": error,
        })

        all_rows.extend(rows)
        stage_rows.append({
            "video_id": video_id, "status": "success", "scene_count": len(rows),
            "stage_signature": stage_sig, "backend": backend,
            "runtime_sec": round(video_runtime, 3), "error": error,
        })

        avg_runtime = sum(completed_durations) / len(completed_durations)
        eta_sec = avg_runtime * (len(videos) - video_number)
        video_bar.update(1)
        video_bar.set_postfix_str(f"done={video_number}/{len(videos)} | scenes={len(all_rows)} | ETA={eta_sec/60:.1f}m")

        write_progress_state({
            "status": "running" if video_number < len(videos) else "finalizing",
            "video_total": len(videos),
            "video_completed": video_number,
            "current_video_id": video_id,
            "scene_count_total": len(all_rows),
            "last_video_runtime_sec": round(video_runtime, 3),
            "average_video_runtime_sec": round(avg_runtime, 3),
            "elapsed_sec": round(tracker.elapsed_total_sec(), 3),
            "eta_sec": round(eta_sec, 3),
        })
finally:
    video_bar.close()

Scene detection:   0%|          | 0/29 [00:00<?, ?video/s]

[2026-07-31T21:00:55.695925+00:00] ▶ BẮT ĐẦU  [video:L21_V001] — 1/29 frames=37849
[2026-07-31T21:01:15.700057+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V001 | elapsed=0.3 phút
[2026-07-31T21:01:35.702121+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V001 | elapsed=0.7 phút
[2026-07-31T21:01:55.703452+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V001 | elapsed=1.0 phút
[2026-07-31T21:02:15.704779+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V001 | elapsed=1.3 phút
[2026-07-31T21:02:22.496116+00:00] ✔ HOÀN TẤT [video:L21_V001] sau 86.8s — scenes=336 backend=transnetv2_pytorch
[2026-07-31T21:02:22.510325+00:00] ▶ BẮT ĐẦU  [video:L21_V002] — 2/29 frames=31720
[2026-07-31T21:02:42.514822+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V002 | elapsed=0.3 phút
[2026-07-31T21:03:02.517188+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V002 | elapsed=0.7 phút
[2026-07-31T21:03:22.523167+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V002 | elapsed=1.

### 4.2 Ghi manifest tổng hợp & đóng gói output

In [11]:
# ---- Task 5: ghi manifest tổng hợp ----
with tracker.stage("write_manifests") as rec:
    atomic_write_jsonl(OUTPUT_ROOT / "manifests" / "scene_manifest.jsonl", all_rows)
    atomic_write_jsonl(OUTPUT_ROOT / "manifests" / "scene_stage_manifest.jsonl", stage_rows)
    atomic_write_json(OUTPUT_ROOT / "model_info.json", {
        "component": "scene_detection",
        "model": "TransNetV2-PyTorch",
        "package_version": version("transnetv2-pytorch"),
        "threshold": THRESHOLD,
        "device": device,
        "weights_path": str(weights),
        "weights_sha256": weights_checksum,
        "pack_version": PACK_VERSION,
    })
    rec.detail = f"{len(all_rows)} scenes / {len(videos)} videos"

total_runtime = tracker.elapsed_total_sec()
atomic_write_json(OUTPUT_ROOT / "_SUCCESS.json", {
    "status": "success",
    "stage": "01_scene_detection",
    "video_count": len(videos),
    "scene_count": len(all_rows),
    "runtime_sec": round(total_runtime, 3),
    "created_at_utc": utc_now(),
})

[2026-07-31T21:37:52.143976+00:00] ▶ BẮT ĐẦU  [write_manifests]
[2026-07-31T21:37:52.233774+00:00] ✔ HOÀN TẤT [write_manifests] sau 0.1s — 8331 scenes / 29 videos


In [12]:
# ---- Task 6: đóng gói zip ----
ZIP_PATH = Path("/kaggle/working/01_scene_detection_output.zip")
with tracker.stage("package_zip", detail=str(ZIP_PATH)):
    create_zip(OUTPUT_ROOT, ZIP_PATH)

write_progress_state({
    "status": "success",
    "video_total": len(videos),
    "video_completed": len(videos),
    "current_video_id": "",
    "scene_count_total": len(all_rows),
    "elapsed_sec": round(tracker.elapsed_total_sec(), 3),
    "eta_sec": 0.0,
})

print({
    "videos": len(videos),
    "scenes": len(all_rows),
    "runtime_min": round(tracker.elapsed_total_sec() / 60, 2),
    "zip": str(ZIP_PATH),
})

[2026-07-31T21:37:52.266694+00:00] ▶ BẮT ĐẦU  [package_zip] — /kaggle/working/01_scene_detection_output.zip
[2026-07-31T21:37:52.340287+00:00] ✔ HOÀN TẤT [package_zip] sau 0.1s — /kaggle/working/01_scene_detection_output.zip
{'videos': 29, 'scenes': 8331, 'runtime_min': 46.12, 'zip': '/kaggle/working/01_scene_detection_output.zip'}


## 5. Bảng tổng kết thời gian chạy từng tác vụ
#
Liệt kê thời gian của các tác vụ pipeline chính (setup, discover, load model,
write manifests, package zip) và thống kê thời gian xử lý video (nhanh nhất,
chậm nhất, trung bình) để đánh giá hiệu năng.

In [13]:
pipeline_tasks = [r for r in tracker.records if not r.name.startswith("video:")]
video_tasks = [r for r in tracker.records if r.name.startswith("video:")]

print("=== Tác vụ pipeline ===")
print(f"{'Tác vụ':30s} {'Trạng thái':10s} {'Thời gian (s)':>14s}")
for r in pipeline_tasks:
    print(f"{r.name:30s} {r.status:10s} {r.duration_sec:14.2f}")

if video_tasks:
    durations = sorted((r.duration_sec for r in video_tasks))
    n = len(durations)
    print("\n=== Thống kê xử lý video ===")
    print(f"Số video       : {n}")
    print(f"Tổng thời gian : {sum(durations)/60:.2f} phút")
    print(f"Trung bình     : {sum(durations)/n:.2f} s/video")
    print(f"Nhanh nhất     : {durations[0]:.2f} s")
    print(f"Chậm nhất      : {durations[-1]:.2f} s")

    slowest = sorted(video_tasks, key=lambda r: r.duration_sec, reverse=True)[:5]
    print("\n5 video chậm nhất:")
    for r in slowest:
        print(f"  {r.name:30s} {r.duration_sec:8.2f}s  | {r.detail}")

print(f"\nTổng thời gian toàn pipeline: {tracker.elapsed_total_sec()/60:.2f} phút")
print(f"Chi tiết đầy đủ từng tác vụ đã được ghi vào: {STAGE_TIMINGS_PATH}")

=== Tác vụ pipeline ===
Tác vụ                         Trạng thái  Thời gian (s)
setup_output                   success              0.01
discover_videos                success             23.22
resolve_weights                success            521.84
load_model                     success              5.21
write_manifests                success              0.09
package_zip                    success              0.07

=== Thống kê xử lý video ===
Số video       : 29
Tổng thời gian : 36.93 phút
Trung bình     : 76.41 s/video
Nhanh nhất     : 58.47 s
Chậm nhất      : 95.72 s

5 video chậm nhất:
  video:L21_V015                    95.72s  | scenes=336 backend=transnetv2_pytorch
  video:L21_V008                    87.18s  | scenes=352 backend=transnetv2_pytorch
  video:L21_V001                    86.80s  | scenes=336 backend=transnetv2_pytorch
  video:L21_V023                    86.80s  | scenes=300 backend=transnetv2_pytorch
  video:L21_V029                    86.07s  | scenes=284 backe

## 6. Xem trạng thái cuối / kiểm tra file tiến độ
`progress.json` luôn chứa tác vụ hiện tại, video hiện tại, số video hoàn thành,
tổng scene, elapsed và ETA. `stage_timings.jsonl` chứa thời gian của **mọi** tác vụ.

In [14]:
if PROGRESS_JSON_PATH.exists():
    print(PROGRESS_JSON_PATH.read_text(encoding="utf-8"))

if PROGRESS_LOG_PATH.exists():
    print("\n===== 20 dòng log cuối =====")
    lines = PROGRESS_LOG_PATH.read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[-20:]))

{
  "stage": "01_scene_detection",
  "updated_at_utc": "2026-07-31T21:37:52.341389+00:00",
  "status": "success",
  "video_total": 29,
  "video_completed": 29,
  "current_video_id": "",
  "scene_count_total": 8331,
  "elapsed_sec": 2767.312,
  "eta_sec": 0.0
}


===== 20 dòng log cuối =====
[2026-07-31T21:33:59.887823+00:00] ▶ BẮT ĐẦU  [video:L21_V029] — 27/29 frames=34689
[2026-07-31T21:34:19.892773+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V029 | elapsed=0.3 phút
[2026-07-31T21:34:39.894775+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V029 | elapsed=0.7 phút
[2026-07-31T21:34:59.897273+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V029 | elapsed=1.0 phút
[2026-07-31T21:35:19.898401+00:00]    …vẫn đang chạy: TransNetV2 đang xử lý L21_V029 | elapsed=1.3 phút
[2026-07-31T21:35:25.956781+00:00] ✔ HOÀN TẤT [video:L21_V029] sau 86.1s — scenes=284 backend=transnetv2_pytorch
[2026-07-31T21:35:25.970333+00:00] ▶ BẮT ĐẦU  [video:L21_V030] — 28/29 frames=31801
[2026-07-31

## 7. Kiểm tra output

In [15]:
with tracker.stage("validate_output"):
    assert all_rows
    by_video: dict[str, list[dict[str, Any]]] = {}
    for row in all_rows:
        by_video.setdefault(row["video_id"], []).append(row)
        assert row["start_frame"] <= row["end_frame"]
        assert row["duration_sec"] >= 0
    for video_id, rows in by_video.items():
        rows.sort(key=lambda item: item["scene_index"])
        assert len({row["scene_id"] for row in rows}) == len(rows)

print("Validation passed for", len(by_video), "videos")

[2026-07-31T21:37:52.492485+00:00] ▶ BẮT ĐẦU  [validate_output]
[2026-07-31T21:37:52.504969+00:00] ✔ HOÀN TẤT [validate_output] sau 0.0s
Validation passed for 29 videos
